In [1]:
import re
from collections import Counter

import numpy as np
import pandas as pd

pattern_ecran = re.compile(r"\((.*?)\)")
pattern_conf_ecran = re.compile(r"<(.*?)>")
pattern_chaine = re.compile(r"\$(.*?)\$")

def read_ds(ds_name: str):
    with open(f"data/{ds_name}.csv") as f:
        max_actions = max((len(str(c).split(",")) for c in f.readlines()))
        f.seek(0)
        _names = ["util", "navigateur"] if "train" in ds_name else ["navigateur"]
        _names.extend(range(max_actions - len(_names)))
        return pd.read_csv(f, names=_names, dtype=str)


def to_categories(df, col="util"):
    df[[col]] = df[[col]].apply(lambda x: x.cat.codes)


def filter_action(value: str):
    for delim in ["(", "<", "$", "1"]:
        if delim in value and (low_ind := value.index(delim)):
            value = value[:low_ind]
    return value


def get_outliers(df):
    df_length = df.iloc[:, 2:].notna().sum(axis=1)
    log_length = np.log10(df_length)
    q1, q3 = log_length.quantile([0.25, 0.75])
    iqr = q3 - q1
    borne_basse, borne_haute = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = (log_length < borne_basse) | (log_length > borne_haute)

    # TODO : add removing duplicate times
    return outliers


def get_sessions(df):
    debut = 2 if "util" in df.columns else 1  # le test n'a pas de colonne util
    return [
        [v for v in ligne if isinstance(v, str)]
        for ligne in df.iloc[:, debut:].to_numpy()
    ]


def is_timestamp(a):
    return re.fullmatch(r"t\d+", a) is not None


def most_frequent(valeurs):
    # renvoie la valeur la plus fréquente de la liste, ou "aucun" si la liste est vide
    return Counter(valeurs).most_common(1)[0][0] if valeurs else "aucun"



In [2]:
features_train = read_ds("train")
features_test = read_ds("test")

features_train["util"] = pd.Categorical(features_train["util"])

uniques = features_train.iloc[:,2:].stack().dropna().unique()
filtered_uniques = list(set([filter_action(un) for un in uniques if not un.startswith("t")]))

train_outliers = get_outliers(features_train)
cleaned_features_train = features_train[~train_outliers].reset_index(drop=True)
y = cleaned_features_train["util"].cat.codes.to_numpy()

sessions_train = get_sessions(cleaned_features_train)
sessions_test = get_sessions(features_test)


In [3]:
def build_features(sessions, filtered_uniques):
    lignes = []
    for s in sessions:
        actions = [a for a in s if not is_timestamp(a)]
        temps = [int(a[1:]) for a in s if is_timestamp(a)]
        n = max(len(actions), 1)
        
        action_counts = []
        current_count = 0
        for a in s:
            if is_timestamp(a):
                action_counts.append(current_count)
                current_count = 0
            else:
                current_count += 1
        # Capture any remaining actions after the last timestamp
        if current_count > 0 or not action_counts:
            action_counts.append(current_count)
            
        # Standard deviation of the action counts between timestamps
        actions_per_interval_std = np.std(action_counts) if len(action_counts) > 1 else 0
        f = {
            "n_actions": n,
            "n_actions_uniques": len(set(actions)),
            "ratio_uniques": len(set(actions)) / n,
            "duree": max(temps, default=0),
            "std": actions_per_interval_std
        }
        
        compte = Counter(filter_action(a) for a in actions)
        for type_action in filtered_uniques:
            f["freq_" + type_action] = compte.get(type_action, 0) / n
        lignes.append(f)

    return pd.DataFrame(lignes)

X_train_core = build_features(sessions_train, filtered_uniques).reset_index(drop=True)
X_test_core = build_features(sessions_test, filtered_uniques).reset_index(drop=True)

In [4]:
def regex_features(sessions):
    lignes = []
    for s in sessions:
        # Extract matches action by action
        ecrans = [m for a in s for m in pattern_ecran.findall(a)]
        confs = [m for a in s for m in pattern_conf_ecran.findall(a)]
        chaines = [m for a in s for m in pattern_chaine.findall(a)]
        
        # Calculate dominance ratio for the top screen (how focused the user is)
        ecran_counts = Counter(ecrans)
        top_ecran_ratio = ecran_counts.most_common(1)[0][1] / len(ecrans) if ecrans else 0.0
        
        conf_counts = Counter(confs)
        top_conf_ratio = conf_counts.most_common(1)[0][1] / len(confs) if confs else 0.0

        lignes.append({
            # Top elements
            "ecran_top": most_frequent(ecrans),
            "conf_top": most_frequent(confs),
            "chaine_top": most_frequent(chaines),
            
            # Flow features (Entry & Exit points)
            "ecran_first": ecrans[0] if ecrans else "aucun",
            "ecran_last": ecrans[-1] if ecrans else "aucun",
            "conf_first": confs[0] if confs else "aucun",
            "conf_last": confs[-1] if confs else "aucun",
            
            # Volume & Diversity features
            "n_ecrans_total": len(ecrans),
            "n_ecrans_distincts": len(set(ecrans)),
            "n_confs_total": len(confs),
            "n_confs_distinctes": len(set(confs)),
            "n_chaines_total": len(chaines),
            "n_chaines_distinctes": len(set(chaines)),
            
            # Behavioral focus ratio
            "top_ecran_ratio": top_ecran_ratio,
            "top_conf_ratio": top_conf_ratio,
        })
        
    return pd.DataFrame(lignes)

X_regex_train = regex_features(sessions_train).reset_index(drop=True)
X_regex_test = regex_features(sessions_test).reset_index(drop=True)

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

cat_features_train = pd.concat([
    cleaned_features_train[["navigateur"]], 
    X_regex_train
], axis=1)

cat_features_test = pd.concat([
    features_test[["navigateur"]], 
    X_regex_test
], axis=1)

categorical_cols = [
    "navigateur", 
    "ecran_top", "conf_top", "chaine_top", 
    "ecran_first", "ecran_last", 
    "conf_first", "conf_last"
]

for col in categorical_cols:
    if col in cat_features_train.columns:
        train_cat = pd.Categorical(cat_features_train[col])
        cat_features_train[col] = train_cat.codes
        
        cat_features_test[col] = pd.Categorical(
            cat_features_test[col], 
            categories=train_cat.categories
        ).codes

X_final_train = pd.concat([X_train_core, cat_features_train], axis=1)
X_final_test = pd.concat([X_test_core, cat_features_test], axis=1)

# Align columns so train and test match perfectly
X_final_test = X_final_test.reindex(columns=X_final_train.columns, fill_value=0)

# 3. Scale numerical features (crucial if you want to test LogisticRegression later, and helps Random Forests too)
scaler = StandardScaler()
X_final_train_scaled = scaler.fit_transform(X_final_train)
X_final_test_scaled = scaler.transform(X_final_test)

# 4. Validation split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_final_train, y, test_size=0.2, random_state=42, stratify=y
)

/tmp/ipykernel_2734/739707615.py:29: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  cat_features_test[col] = pd.Categorical(


In [7]:
X_final_train

,n_actions,n_actions_uniques,ratio_uniques,duree,std,freq_Fermeture d'une dialogue,freq_Sélection d'un élément,freq_Affichage d'une arborescence,freq_Lancement d'un tableau de bord,freq_Affichage d'une erreur,...,conf_first,conf_last,n_ecrans_total,n_ecrans_distincts,n_confs_total,n_confs_distinctes,n_chaines_total,n_chaines_distinctes,top_ecran_ratio,top_conf_ratio
0,2514,53,0.021082,2905,3.661103,0.146778,0.000000,0.000398,0.0,0.002387,...,51,48,191,7,7,3,899,2,0.465969,0.428571
1,90,23,0.255556,230,1.268770,0.022222,0.000000,0.000000,0.0,0.000000,...,111,48,8,5,4,2,43,2,0.375000,0.500000
2,608,84,0.138158,750,3.740875,0.062500,0.016447,0.001645,0.0,0.000000,...,9,101,94,17,35,7,52,2,0.202128,0.371429
3,886,56,0.063205,1445,2.856445,0.047404,0.000000,0.000000,0.0,0.001129,...,99,48,249,10,9,4,203,1,0.469880,0.444444
4,173,37,0.213873,275,2.759070,0.057803,0.000000,0.000000,0.0,0.000000,...,74,101,21,8,7,4,11,2,0.285714,0.285714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3269,122,37,0.303279,135,4.001913,0.065574,0.000000,0.000000,0.0,0.008197,...,111,48,12,6,7,3,50,2,0.250000,0.428571
3270,439,66,0.150342,730,2.962318,0.113895,0.015945,0.002278,0.0,0.009112,...,74,67,81,19,32,9,4,1,0.148148,0.312500
3271,1457,116,0.079616,1730,5.135937,0.059712,0.000000,0.000686,0.0,0.006177,...,51,48,228,15,62,6,367,7,0.342105,0.354839
3272,630,74,0.117460,1150,2.466940,0.068254,0.000000,0.047619,0.0,0.001587,...,74,67,88,16,29,7,0,0,0.193182,0.310345


In [6]:
from sklearn.ensemble import ExtraTreesClassifier
model = ExtraTreesClassifier(
    n_estimators=1000,
    random_state=42,
    n_jobs=-1
)

from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(
    n_estimators=1000,
    random_state=42
)

# from sklearn.ensemble import HistGradientBoostingClassifier
# <- Nullissime
# model = HistGradientBoostingClassifier(
#     max_iter=50,
#     random_state=42
# )

# from sklearn.linear_model import LogisticRegression

# model = LogisticRegression(
#     C=1.0,
#     max_iter=100,
#     solver='saga',
#     random_state=42
# )

model.fit(X_tr, y_tr)
result = model.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(y_val, result):.4f}")
print(f"F1 score: {f1_score(y_val, result, average="macro"):.4f}")

Validation Accuracy: 0.8611
F1 score: 0.8264


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# 1. Create text corpus for all training sessions
corpus_train = ["|".join([filter_action(a) for a in s if not is_timestamp(a)]) for s in sessions_train]

# 2. Split the corpus using the same train/val split logic
X_corpus_tr, X_corpus_val, y_tr, y_val = train_test_split(
    corpus_train, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Build and evaluate KNN Pipeline (n_neighbors=1 looks for exact sequence matches)
knn_pipeline = make_pipeline(
    CountVectorizer(ngram_range=(1, 2), binary=True),
    KNeighborsClassifier(n_neighbors=1, n_jobs=-1)
)

knn_pipeline.fit(X_corpus_tr, y_tr)
result = knn_pipeline.predict(X_corpus_val)

print(f"KNN Validation Accuracy: {accuracy_score(y_val, result):.4f}")
print(f"KNN F1 score: {f1_score(y_val, result, average='macro'):.4f}")

["Création d'un écran Affichage d'une dialogue Exécution d'un bouton Fermeture d'une dialogue Affichage d'une dialogue Exécution d'un bouton Fermeture d'une dialogue Création d'un écran Exécution d'un bouton Affichage d'une dialogue Fermeture d'une dialogue Double-clic Exécution d'un bouton Lancement d'une stat Affichage d'un toast Lancement d'une stat Affichage d'une dialogue Exécution d'un bouton Fermeture d'une dialogue Double-clic Lancement d'une action générique Affichage d'une dialogue Affichage d'un toast Filtrage / Tri Exécution d'un bouton Fermeture d'une dialogue Création d'un écran Exécution d'un bouton Lancement d'une action générique Exécution d'un bouton Affichage d'une dialogue Saisie dans un champ Exécution d'un bouton Fermeture d'une dialogue Affichage d'une dialogue Exécution d'un bouton Fermeture d'une dialogue Affichage d'une dialogue Affichage d'un toast Affichage d'une dialogue Exécution d'un bouton Fermeture d'une dialogue Affichage d'un toast Fermeture d'une dia

In [ ]:
predictions_encoded = model.predict(X_final_test_scaled)
predictions = cleaned_features_train["util"].cat.categories[predictions_encoded]

submission = pd.DataFrame({
    'RowId': range(1, len(X_final_test) + 1), 
    'prediction': predictions
})

# 5. Save to a CSV file (index=False is crucial for Kaggle!)
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' created successfully!")

Submission file 'submission.csv' created successfully!
